In [ ]:
knitr::opts_chunk$set(echo = TRUE, eval = TRUE)



## Intro 

In this final section we incorporate some census data onto our shapefile so we can calculate and explore the differences between crime rate and crime counts. We then introduce cartograms as an alternative thematic map. 

In theory showing that crime rate accounts for population differences and is a more meaningful measure than raw counts.

## Load packages

As always the first step is to load the necessary R packages via the library function. If you do not have these packages installed then please follow the instructions in the *Preliminary Task.Rmd* file. 

## Load Packages


In [ ]:
# for data reading/manipulation 
library(dplyr)
library(tidyr)
library(readr)
library(tibble)
library(janitor)
library(readxl)
# for spatial data and gis
library(sf)
library(ggmap)
library(ggplot2)
library(ggspatial)
library(ggspatial)
library(spdep)
library(leaflet) 
library(RColorBrewer)
library(tmap)


## Read in the population statistics

Count data is not entirely accurate of population density. Whilst the code in Section_1 and Section_2 might help us identify interesting patterns, point-level open crime data is rarely used in isolation for detailed analysis. 

For one thing, the data are points are geomasked. This means that points are highly likely to be overlapped, giving a skewed picture of the distribution. There are ways round this, such as through jittering or applying census based data. To view more information about jittering please view the 'Additional Topic.rmd' file,


### Census 2022

We will be using the new census 2022 data to calculate the crime rate across both residential population vs working population,

The data was downloaded from CKAN which can be accessed from the front page of our website *https://statistics.ukdataservice.ac.uk/dataset/?vocab_Area_type=Lower%20Super%20Output%20Areas*

I simply searched for 'residents' and 'work' as I want to calculate the difference in crime counts compared to the usual resident population and then to the usual workday population. Download the super layer output areas and once opened in excel, you can select just the rows that correspond to 'Surrey Heath'.

But have no fear if you don't know how to do this or are simply feeling a little lazy (I get it), you can just read in the data that has been cleaned for you in the Data folder. There is more information via the *Dowloading the Data* doc if you are interested in the download process. 

So lets read in the data and use the clean_names() function again to lowercase and tidy all variable names. 

We are going to start with just the resident population which is the data set named 'res_count.xlsx'


In [ ]:
url <- "https://raw.githubusercontent.com/UKDataServiceOpen/Mapping_Crime_Data_R_2026/main/data/census_data/TS001_resident_population.xlsx"

# Download the Excel file to a temporary location before reading
temp_file <- tempfile(fileext = ".xlsx")
download.file(url, temp_file, mode = "wb")

residential_count <- read_excel(temp_file) %>% janitor::clean_names() %>%
  slice(-c(1:7)) %>%
  slice(-c(56, 57)) %>% 
  rename(lsoa = x2, #rename the variables 
         name = ts001_number_of_usual_residents_in_households_and_communal_establishments,
         res_count = x3) 

head(residential_count)


## Join the data to our new shapefile

The next step is to join these new statistics to our previously created shapefile named 'surrey_lsoa". We do thus by using the left_join() function in the dplyr package and attach them by the same LSOA codes. 


In [ ]:
surrey_lsoa <- left_join(surrey_lsoa, residential_count, by = c("lsoa21cd"="lsoa"))

head(surrey_lsoa)


Now you will see the census data has merged into the shapefile, Great lets move on to calculating crime rate.



## How to calculate the crime rate?

A crime rate is calculated by dividing the number of reported crimes by the total population, and then multiplying by 100,000. 

So for our dataset, we take the count variable, divide by the 'pop' variable (workday or residential), and then times by 1000 (in this instance we use 1000 as this is the average population of an LSOA, if you were using larger UoA you can choose to multiply by 100,000. Just remember what affect this will have on your rate and how this then interprets across your results.

In order to work out the crime rate, we need to create a new variable that takes the count/pop*1000. We can use the mutate() function from the dplyr package to calculate this for us. 


In [ ]:
surrey_lsoa <- surrey_lsoa %>% 
  mutate(crime_rate = (count/res_count*1000))
         
head(surrey_lsoa)


## Now lets explore these trends using ggplot and tmaps;


### First ggplot


In [ ]:
ggplot() + 
  annotation_map_tile() + 
  geom_sf(data = surrey_lsoa, aes(fill = crime_rate), alpha = 0.5) + 
  scale_fill_gradient2(name ="Crime Rate")

##Improve Mapping
ggplot(surrey_lsoa) + 
  geom_sf(aes(fill = crime_rate), color = "black", lwd = 0.2) +
  scale_fill_viridis_c(option = "magma", name = "Crime Rate per 10,000") +
  labs(title = "Crime Rate by LSOA in Surrey",
       caption = "Source: Police.uk & Census 2022") +
  theme_minimal()


### What about tmaps 



In [ ]:
tm_shape(surrey_lsoa) + 
  tm_fill("crime_rate", style = "quantile") + 
  tm_borders(alpha = 0.3)

##Improved Mapping 
tmap_mode("plot")

tm_shape(surrey_lsoa) + 
  tm_polygons("crime_rate", style = "jenks", palette = "Reds", 
              title = "Crime Rate (per 10,000)") + 
  tm_borders("black", lwd = 0.5) +
  tm_layout(main.title = "Crime Rate by LSOA in Surrey")


## Multivariate Mapping (using urban vs rural classification)



In [ ]:
# Read and join urban/rural classification (multivariate mapping)
url <- "https://raw.githubusercontent.com/UKDataServiceOpen/Mapping_Crime_Data_R_2026/main/data/census_data/urban_rural.csv"

# Download the Excel file to a temporary location before reading
temp_file <- tempfile(fileext = ".csv")
download.file(url, temp_file, mode = "wb")

urban_rural <- read_csv(temp_file) %>%  # or your URL
  janitor::clean_names() %>%
  select(lsoa21cd, urban_rural_flag, ruc21nm) %>%
  rename(rc_cat = ruc21nm) %>%
  mutate(urban_rural_flag = recode(urban_rural_flag, 
                                   "U" = "Urban", 
                                   "R" = "Rural"))
  

# Join to our shapefile
surrey_lsoa <- left_join(surrey_lsoa, urban_rural, by = "lsoa21cd")

head(surrey_lsoa)


Lets first explore the binary variable 'urban_rural_flag':



In [ ]:
# Multivariate map: crime rate *by* urban/rural
ggplot(surrey_lsoa) +
  geom_sf(aes(fill = crime_rate, colour = urban_rural_flag), size = 0.5) +
  scale_fill_viridis_c(name = "Crime Rate\n(per 1,000)") +
  scale_colour_manual(values = c("Urban" = "black", "Rural" = "white")) +
  labs(title = "Crime Rate by LSOA, Coloured by Urban/Rural",
       subtitle = "Multivariate mapping: outcome (rate) across a second variable (urban/rural)") +
  theme_minimal()

# Or small multiples with tmap
tmap_mode("view")
tm_shape(surrey_lsoa) +
  tm_fill("crime_rate", style = "quantile", palette = "Reds") +
  tm_facets(by = "urban_rural_flag") +
  tm_layout(title = "Crime Rates: Urban vs Rural LSOAs") +
  tm_borders(alpha = 0.3)


What about for the more detailed category 'rc_cat':



In [ ]:
# Detailed categories (bonus)
ggplot(surrey_lsoa) +
  geom_sf(aes(fill = crime_rate, colour = rc_cat), size = 0.4) +
  scale_fill_viridis_c(name = "Crime Rate\n(per 1k)") +
  scale_colour_brewer(type = "qual", palette = "Set1", na.value = "grey") +
  labs(title = "Crime Rate by LSOA: Detailed Urban/Rural Types",
       subtitle = "Multivariate: rate (fill) vs urban/rural category (borders)") +
  theme_minimal() +
  theme(legend.position = "bottom")

tmap_mode("view")
tm_shape(surrey_lsoa) +
  tm_fill("crime_rate", style = "jenks", palette = "Reds", 
          title = "Crime Rate\n(per 1k)") +
  tm_facets(by = "rc_cat", free.coords = FALSE) +
  tm_borders("black", lwd = 0.3, alpha = 0.6) +
  tm_layout(main.title = "Crime Rates by Detailed Urban/Rural Classification",
            legend.position = c("left", "bottom"))


# Activity 3

We have successfully mapped the residential population, now lets do the same with the variable employement. This is for all usual residents who are within employment in the area.

First step is to read and clean the data as seen here:


In [ ]:
url <- "https://raw.githubusercontent.com/UKDataServiceOpen/Mapping_Crime_Data_R_2026/main/data/census_data/TS006_density_population.xlsx"

# Download the Excel file to a temporary location before reading
temp_file <- tempfile(fileext = ".xlsx")
download.file(url, temp_file, mode = "wb")

employement_pop <- read_excel(temp_file) %>% janitor::clean_names() %>%
  slice(-c(1:7)) %>%
  slice(-c(56, 57)) %>% 
  rename(lsoa = wd102ew_population_density, #rename the variables 
         name = x2,
         emp_count = x3) 

head(employement_pop)


And then remember you need to join this to our shapefile by using the left_join function.



In [ ]:
surrey_lsoa <- left_join(surrey_lsoa, employement_pop, by = c("lsoa21cd"="lsoa"))

head(surrey_lsoa)


So now you've added the work population count, it time to calculate the crime rate. Follow the steps below to try and complete the activity. 


Steps: 

1  First calculate the crime rate 
2. Plot using ggplot 
3. Plot using tmap 
4. Plot both maps (residential and workday) together using tmap_arrange 
5. Plot a cartogram of residential population

Is there a difference between the crime rate when using employement population compared to residential population? Would we expect to see these trends?


In [ ]:
# 1) First calculate the crime rate, assign it a new variable and named it crimerate2

surrey_lsoa <- surrey_lsoa %>% 
  mutate(crime_rate2 = (count/......)*...)


#2) Plot using ggplot 

ggplot() + 
  annotation_map_tile() + 
  geom_sf(data = ....., aes(fill = ......), alpha = 0.5) + 
  scale_fill_gradient2(name ="Crime Rate")


#3) Plot using tmap 

tm_shape(surrey_lsoa) + 
  tm_fill(....., style = "quantile") + 
  tm_borders(alpha = 0.3)


#4) Compare the workday vs residential population 

e <- tm_shape(surrey_lsoa) + 
  tm_fill(....., style = "quantile", title = "Workday Pop") + 
  tm_borders(alpha = 0.3)

f <- tm_shape(surrey_lsoa) + 
  tm_fill(....., style = "quantile", title = "Residential pop") + 
  tm_borders(alpha = 0.3)


tmap_arrange(...., .....)


#5 employment rates by urban/rural (multivariate!) 

tm_shape(surrey_lsoa) +
  tm_fill(....., style = "jenks", palette = "Blues") +
  tm_facets(by = .....) +
  tm_borders("black", lwd = 0.3) +
  tm_layout(title = "Workday Crime Rates: Urban vs Rural")


 


Data Citation: 

1.	UK Police API. (2026). Street-level crime data. data.police.uk. Retrieved February 27, 2026, from https://data.police.uk
2.	UK Data Service Census Support. (2021). England LSOA boundaries 2021. Retrieved from https://borders.ukdataservice.ac.uk 
3.	Office for National Statistics. (2022). Census 2021 Table TS001: Number of usual residents. Nomis/UK Data Service. Retrieved from https://statistics.ukdataservice.ac.uk
4.	Office for National Statistics. (2022). Census 2021 Table TS066: Economic acitivty. Nomis/UK Data Service. Retrieved from https://statistics.ukdataservice.ac.uk
5.	Office for National Statistics. (2021). Rural Urban Classification (2021) of LSOAs in England and Wales. Open Geography Portal. Retrieved from https://geoportal.statistics.gov.uk 
